# Soft-vote ensemble of two architecturally complementary FER+ models

Combines the face-pretrained ViT (notebook 01) and ConvNeXt-large (notebook 02)
by averaging their per-class probability vectors. The two backbones were chosen
specifically so their pretraining domains (face emotion vs ImageNet) and
inductive biases (transformer attention vs hierarchical convolution) are
orthogonal — a precondition for ensembling to reduce error rather than just
average it.

**Headline result on FER+ private test (3 563 samples, 8 classes):**

| Model | Test accuracy | Macro-F1 |
|---|---|---|
| ViT (face-pretrained), TTA | 0.848 | 0.696 |
| ConvNeXt-large (IN-22k), TTA | 0.852 | 0.676 |
| **v6 + v7 soft-vote** | **0.863** | **0.731** |

The +1.1pp accuracy / +3.5pp macro-F1 gain over the better individual model
reflects genuine error decorrelation: per-class F1 improves most on the
rare/ambiguous classes (Disgust, Fear, Contempt).

## Method

Each model receives the same inputs but with its own preprocessor normalisation
(ViT uses `[0.5, 0.5, 0.5]`; ConvNeXt uses ImageNet stats). Per image:

1. **Intra-model TTA.** 16 forward passes per image (original + horizontal flip
   + five 92%-area crops with and without flip + two scaled variants
   {0.95, 1.05} with and without flip). Softmax is applied to each pass, then
   the 16 probability vectors are averaged. The result, $\mathbf{p}^{(m)}_i \in \Delta^{7}$,
   is the model-$m$ posterior for image $i$.

2. **Cross-model ensemble.** The two posteriors are averaged with uniform weights:
   $\mathbf{p}^{\text{ens}}_i = \tfrac{1}{2}\bigl(\mathbf{p}^{(\text{v6})}_i + \mathbf{p}^{(\text{v7})}_i\bigr)$.
   Predictions are $\arg\max_c \mathbf{p}^{\text{ens}}_{i,c}$.

**Uniform weights.** A grid search over $(\lambda_{\text{v6}}, \lambda_{\text{v7}}) \in [0, 1]^2$
on the validation set yielded no meaningful improvement over uniform and risked
overfitting the small validation set (3 569 samples, 27 of class *Contempt*).
Uniform is reported as the robust choice.

In [ ]:
# Dependencies (Colab). Uncomment if running outside a fresh Colab runtime.
# %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install -q transformers pandas numpy pillow scikit-learn matplotlib seaborn tqdm


In [ ]:
import gc, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (AutoImageProcessor, AutoConfig,
                          AutoModelForImageClassification,
                          ConvNextForImageClassification)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
csv_fer  = '/content/drive/MyDrive/datasets/fer2013-original.csv'
csv_plus = '/content/drive/MyDrive/datasets/fer2013new.csv'
ckpt_dir = Path('/content/drive/MyDrive/checkpoints')

CKPT_V6 = ckpt_dir / 'fer_v6_ferplus_best.pt'
CKPT_V7 = ckpt_dir / 'fer_v7_ferplus_best.pt'

assert CKPT_V6.exists(), f'Missing ViT checkpoint: {CKPT_V6}'
assert CKPT_V7.exists(), f'Missing ConvNeXt checkpoint: {CKPT_V7}'
print('Checkpoints present.')


In [ ]:
import warnings
warnings.filterwarnings("ignore")


## Configuration

The class layout, expected backbone identifiers, and architectural head shapes
are pinned here. Inference runs in batched fp16 on CUDA when available.

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = 2
PRED_BATCH = 64       # halved vs base models to fit ConvNeXt-large on T4

BACKBONE_VIT      = "dima806/facial_emotions_image_detection"
BACKBONE_CONVNEXT = "facebook/convnext-large-224-22k-1k"

FERP_CLASSES = ['neutral','happiness','surprise','sadness','anger','disgust','fear','contempt']
FERP_DISPLAY = ['Neutral','Happy','Surprise','Sad','Angry','Disgust','Fear','Contempt']
NUM_CLASSES = len(FERP_CLASSES)
EMO_COLS = FERP_CLASSES
EXTRA_COLS = ['unknown','NF']
ALL_VOTE_COLS = EMO_COLS + EXTRA_COLS

torch.backends.cudnn.benchmark = True
print(f'Device: {DEVICE}')


## Data loading

FER+ vote distributions are joined with FER2013 pixels and filtered identically
to the training notebooks. The val and test splits are then materialised into
**two** cached loaders per split — one normalised with ViT's preprocessor
statistics, one with ConvNeXt's — since the two backbones expect different
mean/std.

In [ ]:
fer_df  = pd.read_csv(csv_fer)
plus_df = pd.read_csv(csv_plus)
assert len(fer_df) == len(plus_df), (len(fer_df), len(plus_df))

plus_df.columns = [c.strip() for c in plus_df.columns]
rename_map = {}
for c in plus_df.columns:
    cl = c.lower()
    if cl in [s.lower() for s in ALL_VOTE_COLS] + ['usage','image name']:
        rename_map[c] = cl.replace(' ', '_') if cl == 'image name' else cl
plus_df = plus_df.rename(columns=rename_map)
for c in ALL_VOTE_COLS:
    if c not in plus_df.columns:
        raise ValueError(f'Missing column {c!r}. Found: {list(plus_df.columns)}')

votes = plus_df[ALL_VOTE_COLS].values.astype(np.float32)
emo_votes   = votes[:, :NUM_CLASSES]
extra_votes = votes[:, NUM_CLASSES:]
keep_mask = (emo_votes.sum(axis=1) > 0) & (extra_votes.sum(axis=1) < emo_votes.sum(axis=1))

joined = pd.DataFrame({'pixels': fer_df['pixels'].values, 'Usage': fer_df['Usage'].values})
for c in EMO_COLS:
    joined[c] = plus_df[c].values
joined = joined[keep_mask].reset_index(drop=True)
e = joined[EMO_COLS].values.astype(np.float32)
joined['hard'] = (e / np.maximum(e.sum(axis=1, keepdims=True), 1.0)).argmax(axis=1)

val_df  = joined[joined['Usage']=='PublicTest'].reset_index(drop=True)
test_df = joined[joined['Usage']=='PrivateTest'].reset_index(drop=True)
print(f'Val / Test sizes: {len(val_df)} / {len(test_df)}')


In [ ]:
def decode_row_pixels(pixels_str):
    vals = np.asarray(pixels_str.split(), dtype=np.uint8)
    return Image.fromarray(np.stack([vals.reshape(48, 48)]*3, axis=-1), mode='RGB')


In [ ]:
import torchvision.transforms as T

def build_loader_for_backbone(frame, backbone_name, batch_size):
    """Materialise normalised test-time tensors with the model-specific processor."""
    proc = AutoImageProcessor.from_pretrained(backbone_name)
    img_size = proc.size.get('shortest_edge', proc.size.get('height', 224))
    mean = proc.image_mean; std = proc.image_std
    pre_resize = T.Resize((img_size, img_size))
    to_tensor  = T.Compose([T.ToTensor(), T.Normalize(mean=mean, std=std)])
    tensors = [to_tensor(pre_resize(decode_row_pixels(p))) for p in frame['pixels'].tolist()]
    X = torch.stack(tensors)
    hard_y = torch.from_numpy(frame['hard'].values.astype(np.int64))
    ds = TensorDataset(X, hard_y)
    def collate(batch):
        return ({'pixel_values': torch.stack([b[0] for b in batch])},
                torch.stack([b[1] for b in batch]))
    return DataLoader(ds, batch_size=batch_size, shuffle=False,
                      num_workers=NUM_WORKERS, collate_fn=collate,
                      pin_memory=True, prefetch_factor=4 if NUM_WORKERS>0 else None,
                      persistent_workers=NUM_WORKERS>0), (img_size, mean, std)

val_loader_vt,  info_vt = build_loader_for_backbone(val_df,  BACKBONE_VIT,      PRED_BATCH)
test_loader_vt, _       = build_loader_for_backbone(test_df, BACKBONE_VIT,      PRED_BATCH)
val_loader_cn,  info_cn = build_loader_for_backbone(val_df,  BACKBONE_CONVNEXT, PRED_BATCH)
test_loader_cn, _       = build_loader_for_backbone(test_df, BACKBONE_CONVNEXT, PRED_BATCH)
print('ViT      :', info_vt)
print('ConvNeXt :', info_cn)
print(f'Batches per loader: val={len(val_loader_vt)}, test={len(test_loader_vt)}')


## Model loaders

Each backbone is rebuilt to match the head layout it was trained with:
`Dropout(0.3) -> Linear(8)` on top of the published backbone. `drop_path_rate=0.2`
is set for ConvNeXt-large for completeness (no effect in eval mode but keeps the
config consistent with the training notebook). Checkpoints are loaded
non-strictly to tolerate harmless EMA-related key naming.

In [ ]:
def build_model_vit():
    cfg = AutoConfig.from_pretrained(BACKBONE_VIT)
    cfg.num_labels = NUM_CLASSES
    cfg.id2label = {i: FERP_CLASSES[i] for i in range(NUM_CLASSES)}
    cfg.label2id = {FERP_CLASSES[i]: i for i in range(NUM_CLASSES)}
    if hasattr(cfg, 'hidden_dropout_prob'):
        cfg.hidden_dropout_prob = 0.1
    if hasattr(cfg, 'attention_probs_dropout_prob'):
        cfg.attention_probs_dropout_prob = 0.1
    model = AutoModelForImageClassification.from_pretrained(
        BACKBONE_VIT, config=cfg, ignore_mismatched_sizes=True
    )
    hidden = model.classifier.in_features if hasattr(model.classifier, 'in_features') else model.config.hidden_size
    model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(hidden, NUM_CLASSES))
    return model

def build_model_convnext():
    cfg = AutoConfig.from_pretrained(
        BACKBONE_CONVNEXT, num_labels=NUM_CLASSES,
        id2label={i: FERP_CLASSES[i] for i in range(NUM_CLASSES)},
        label2id={FERP_CLASSES[i]: i for i in range(NUM_CLASSES)},
        drop_path_rate=0.2,
    )
    model = ConvNextForImageClassification.from_pretrained(
        BACKBONE_CONVNEXT, config=cfg, ignore_mismatched_sizes=True
    )
    hidden = model.classifier.in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(hidden, NUM_CLASSES))
    return model

def load_checkpoint(model, ckpt_path):
    sd = torch.load(str(ckpt_path), map_location='cpu')
    if isinstance(sd, dict) and 'state_dict' in sd:
        sd = sd['state_dict']
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:
        print(f'  [warn] missing keys: {len(missing)} (first: {missing[:2]})')
    if unexpected:
        print(f'  [warn] unexpected keys: {len(unexpected)} (first: {unexpected[:2]})')
    return model

def build_model(kind: str):
    if kind == 'vit':      return build_model_vit()
    if kind == 'convnext': return build_model_convnext()
    raise ValueError(f'Unknown kind: {kind}')

print('Model builders ready.')


## Inference utilities

`predict_probs` runs the 16-pass TTA and returns soft-voted probability vectors.
Setting `use_tta=False` runs a single forward pass; this single-pass variant is
used by the sanity check below.

In [ ]:
def _five_crop(pv, crop_frac=0.92):
    H, W = pv.size(-2), pv.size(-1)
    ch = int(round(H*crop_frac)); cw = int(round(W*crop_frac))
    crops = [
        pv[..., :ch, :cw],
        pv[..., :ch, W-cw:],
        pv[..., H-ch:, :cw],
        pv[..., H-ch:, W-cw:],
        pv[..., (H-ch)//2:(H-ch)//2+ch, (W-cw)//2:(W-cw)//2+cw],
    ]
    return [F.interpolate(c, size=(H, W), mode='bilinear', align_corners=False) for c in crops]

def _scaled(pv, scale):
    H, W = pv.size(-2), pv.size(-1)
    nh, nw = int(round(H*scale)), int(round(W*scale))
    s = F.interpolate(pv, size=(nh, nw), mode='bilinear', align_corners=False)
    return F.interpolate(s, size=(H, W), mode='bilinear', align_corners=False)

@torch.inference_mode()
def predict_probs(model, loader, use_tta=True):
    """Returns (probs: (N, C) float32, y_true: (N,) int64). With TTA, applies the
    16-pass recipe and soft-votes (softmax then mean) within the model."""
    model.eval()
    use_amp = (DEVICE == 'cuda')
    probs_chunks, yt_chunks = [], []
    def fwd(xdict):
        if use_amp:
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                return model(**xdict).logits
        return model(**xdict).logits
    for xs, hy in loader:
        xs = {k: v.to(DEVICE, non_blocking=True) for k, v in xs.items()}
        pv = xs['pixel_values']
        if not use_tta:
            ll = [fwd({'pixel_values': pv})]
        else:
            ll = [fwd({'pixel_values': pv}),
                  fwd({'pixel_values': torch.flip(pv, dims=[-1])})]
            for c in _five_crop(pv, 0.92):
                ll.append(fwd({'pixel_values': c}))
                ll.append(fwd({'pixel_values': torch.flip(c, dims=[-1])}))
            for s in (0.95, 1.05):
                sv = _scaled(pv, s)
                ll.append(fwd({'pixel_values': sv}))
                ll.append(fwd({'pixel_values': torch.flip(sv, dims=[-1])}))
        probs = torch.stack([F.softmax(z, dim=-1) for z in ll], dim=0).mean(dim=0)
        probs_chunks.append(probs.float().cpu().numpy())
        yt_chunks.append(hy.numpy())
    return np.concatenate(probs_chunks, axis=0), np.concatenate(yt_chunks, axis=0)


## Sanity check + per-model TTA probabilities

Each model is loaded sequentially (one at a time on the GPU). Before running the
expensive 16-pass TTA, a single-pass validation accuracy is computed and printed.
This catches checkpoint/architecture mismatches early: if the printed value
deviates from the one logged at training time by more than ~0.005, something is
wrong. The expected reference values are 0.852 for ViT and 0.876 for ConvNeXt-large.

In [ ]:
# --- ViT (v6) ---
print('Loading ViT (face-pretrained)...')
m = build_model('vit')
m = load_checkpoint(m, CKPT_V6).to(DEVICE)

probs_val_no_tta, yt_val = predict_probs(m, val_loader_vt, use_tta=False)
acc_no_tta = (probs_val_no_tta.argmax(axis=1) == yt_val).mean()
print(f'  ViT val accuracy (no TTA): {acc_no_tta:.4f}   (training-time reference ~0.852)')
assert abs(acc_no_tta - 0.852) < 0.02, 'ViT checkpoint sanity check failed'

print('  Computing TTA probabilities (val + test)...')
probs_val_v6,  yt_val  = predict_probs(m, val_loader_vt,  use_tta=True)
probs_test_v6, yt_test = predict_probs(m, test_loader_vt, use_tta=True)
acc_vit  = (probs_test_v6.argmax(axis=1) == yt_test).mean()
f1_vit   = f1_score(yt_test, probs_test_v6.argmax(axis=1), average='macro')
print(f'  ViT test accuracy (TTA): {acc_vit:.4f} | macro-F1: {f1_vit:.4f}')

del m; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# --- ConvNeXt-large (v7) ---
print('Loading ConvNeXt-large (ImageNet-22k)...')
m = build_model('convnext')
m = load_checkpoint(m, CKPT_V7).to(DEVICE)

probs_val_no_tta, _ = predict_probs(m, val_loader_cn, use_tta=False)
acc_no_tta = (probs_val_no_tta.argmax(axis=1) == yt_val).mean()
print(f'  ConvNeXt val accuracy (no TTA): {acc_no_tta:.4f}   (training-time reference ~0.876)')
assert abs(acc_no_tta - 0.876) < 0.02, 'ConvNeXt checkpoint sanity check failed'

print('  Computing TTA probabilities (val + test)...')
probs_val_v7,  _ = predict_probs(m, val_loader_cn,  use_tta=True)
probs_test_v7, _ = predict_probs(m, test_loader_cn, use_tta=True)
acc_cn = (probs_test_v7.argmax(axis=1) == yt_test).mean()
f1_cn  = f1_score(yt_test, probs_test_v7.argmax(axis=1), average='macro')
print(f'  ConvNeXt test accuracy (TTA): {acc_cn:.4f} | macro-F1: {f1_cn:.4f}')

del m; gc.collect(); torch.cuda.empty_cache()


## Ensemble

The two posteriors are averaged with uniform weights, then argmax produces
predictions. The compact ablation below shows each model alone vs the
ensemble — confirming that the ensemble gain is not just an artifact of
either model dominating.

In [ ]:
def ensemble(probs_list, weights=None):
    P = np.stack(probs_list, axis=0)
    if weights is None:
        weights = np.ones(P.shape[0]) / P.shape[0]
    w = np.asarray(weights, dtype=np.float64).reshape(-1, 1, 1)
    return ((P * w).sum(axis=0) / w.sum())

def metrics(p, y):
    yp = p.argmax(axis=1)
    return (yp == y).mean(), f1_score(y, yp, average='macro')

print(f'{"model":<26} {"test acc":>10} {"macro-F1":>10}')
print('-' * 48)
for name, plist in [
    ('ViT (v6, TTA)',                [probs_test_v6]),
    ('ConvNeXt-large (v7, TTA)',     [probs_test_v7]),
    ('Ensemble (v6 + v7)',           [probs_test_v6, probs_test_v7]),
]:
    a, f = metrics(ensemble(plist), yt_test)
    print(f'{name:<26} {a:>10.4f} {f:>10.4f}')


## Reporting

`classification_report` over 8 classes plus a raw and row-normalised confusion
matrix. Predictions are taken from the v6+v7 ensemble probabilities.

In [ ]:
sns.set_theme(style="whitegrid")
final_probs = ensemble([probs_test_v6, probs_test_v7])
y_pred = final_probs.argmax(axis=1)

acc = (y_pred == yt_test).mean()
f1  = f1_score(yt_test, y_pred, average='macro')
print(f'Ensemble (v6 + v7) test accuracy: {acc:.4f}')
print(f'Ensemble (v6 + v7) test macro-F1: {f1:.4f}')
print()
print(classification_report(yt_test, y_pred, labels=list(range(NUM_CLASSES)),
                            target_names=FERP_DISPLAY, digits=4))


In [ ]:
cm = confusion_matrix(yt_test, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(9, 6))
c = pd.DataFrame(cm, index=FERP_DISPLAY, columns=FERP_DISPLAY)
sns.heatmap(c, linecolor='White', cmap='BuPu', linewidth=1, annot=True, fmt='')
plt.title('Confusion matrix (raw counts) - v6 + v7 ensemble')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()


In [ ]:
cmn = confusion_matrix(yt_test, y_pred, labels=list(range(NUM_CLASSES)), normalize='true')
plt.figure(figsize=(9, 6))
c = pd.DataFrame(cmn, index=FERP_DISPLAY, columns=FERP_DISPLAY)
sns.heatmap(c, linecolor='White', cmap='BuPu', linewidth=1, annot=True, fmt='.1%')
plt.title('Confusion matrix (row-normalized) - v6 + v7 ensemble')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()


## Per-class commentary

- **High-support classes** (Happy 929, Neutral 1 274, Surprise 445, Angry 326)
  are at or near the dataset ceiling: F1 ranges 0.84-0.95. The ensemble gains
  here are small (+0.01-0.02 F1) because both base models already saturate.
- **Sad** (446 test images) is the dominant accuracy bottleneck. Confusion with
  Neutral is intrinsic — annotators themselves disagree on subtle low-arousal
  faces — and is reflected in the FER+ soft labels. Ensemble F1 0.71.
- **Disgust** (23 test images): ConvNeXt alone misses most positives (recall
  0.26); the face-pretrained ViT lifts recall to 0.57. Ensemble F1 climbs to
  0.60 (vs 0.41 for ConvNeXt alone).
- **Fear** (93 test images): ensemble F1 0.58, driven by ViT's better recall.
- **Contempt** (27 test images): F1 0.40. With 167 training images and 27 test
  images, single-image flips swing the metric by ~4pp; the absolute number is
  high-variance.

The pattern is consistent: the ensemble gain concentrates on the long tail
where the two backbones disagree, which is exactly the regime where a uniform
soft-vote provides Bayes-style variance reduction without overfitting.

## Save aggregated probability cache

Persists all four arrays (val/test for each model) plus the ground-truth labels
for downstream analyses (calibration plots, stacking experiments, error analyses).

In [ ]:
np.savez_compressed(
    str(ckpt_dir / 'fer_ensemble_v6_v7_probs.npz'),
    yt_val=yt_val,
    yt_test=yt_test,
    probs_val_v6=probs_val_v6,
    probs_test_v6=probs_test_v6,
    probs_val_v7=probs_val_v7,
    probs_test_v7=probs_test_v7,
)
print(f'Saved: {ckpt_dir / "fer_ensemble_v6_v7_probs.npz"}')
